# Click-to-segment: training results

Interactive object segmentation on ADE20K. The user clicks an object, the model returns
that instance's mask. A UNet-style encoder-decoder trained from scratch, no pretrained
backbone, taking 5 input channels: RGB plus a positive and a negative click map.

This notebook reads the artefacts produced by `scripts/train_full.py` and reports what
the trained model actually does.

In [ ]:
import json, sys, os
from pathlib import Path

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

# $HOME differs between MetaCentrum nodes, so try the known locations rather
# than trusting "~" to resolve to the storage the data actually lives on.
CANDIDATE_ROOTS = [
    Path("/storage/brno2/home") / os.environ.get("USER", "") / "projects/ade20k-reference/dataset/ADE20K_2021_17_01/images/ADE",
    Path.home() / "projects/ade20k-reference/dataset/ADE20K_2021_17_01/images/ADE",
]
DATA_ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), None)

OUTPUTS = REPO / "outputs"
print("repo:     ", REPO)
print("data root:", DATA_ROOT)
print("outputs:  ", OUTPUTS)

## Training history

In [ ]:
with open(OUTPUTS / "history.json") as f:
    results = json.load(f)

# train_full.py writes a bare list while running and a summary dict at the end.
history = results["history"] if isinstance(results, dict) else results
summary = results if isinstance(results, dict) else {}

epochs    = [h["epoch"]      for h in history]
train_loss= [h["train_loss"] for h in history]
val_loss  = [h["val_loss"]   for h in history]
train_iou = [h["train_iou"]  for h in history]
val_iou   = [h["val_iou"]    for h in history]

best_epoch = summary.get("best_epoch") or epochs[val_iou.index(max(val_iou))]
best_val   = summary.get("best_val_iou", max(val_iou))

print(f"epochs trained : {len(history)}")
print(f"best epoch     : {best_epoch}")
print(f"best val IoU   : {best_val:.4f}")
if "test_iou" in summary:
    print(f"test IoU       : {summary['test_iou']:.4f}")
print(f"mean epoch time: {sum(h['seconds'] for h in history)/len(history):.0f}s")

### Loss and IoU curves

The dashed line marks the epoch with the best validation IoU. That epoch's weights are
what `best.pt` holds and what the test set is scored against, so a late-training dip
cannot contaminate the reported result.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_loss, ax_iou) = plt.subplots(1, 2, figsize=(13, 4.5))

ax_loss.plot(epochs, train_loss, label="train")
ax_loss.plot(epochs, val_loss, label="validation")
ax_loss.axvline(best_epoch, ls="--", c="gray", lw=1)
ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("BCE + Dice loss")
ax_loss.set_title("Loss"); ax_loss.legend(); ax_loss.grid(alpha=.3)

ax_iou.plot(epochs, train_iou, label="train")
ax_iou.plot(epochs, val_iou, label="validation")
ax_iou.axvline(best_epoch, ls="--", c="gray", lw=1)
ax_iou.scatter([best_epoch], [best_val], zorder=5, c="crimson",
               label=f"best (epoch {best_epoch})")
ax_iou.set_xlabel("epoch"); ax_iou.set_ylabel("IoU")
ax_iou.set_title("Intersection over Union"); ax_iou.legend(); ax_iou.grid(alpha=.3)

fig.tight_layout()
plt.show()

### Reading the curves

Training IoU keeps climbing while validation flattens, and the gap between them widens
in the later epochs. That gap is the model beginning to memorise the training set rather
than learning to generalise, which is why the best epoch is selected on validation and
not simply taken from the final epoch.

The practical consequence: training for longer would not help. More data would.

## Results

In [ ]:
final = history[-1]
best  = next(h for h in history if h["epoch"] == best_epoch)

rows = [
    ("train (final epoch)",      final["train_loss"], final["train_iou"]),
    (f"validation (epoch {best_epoch})", best["val_loss"], best["val_iou"]),
]
if "test_iou" in summary:
    rows.append(("test (held out)", summary["test_loss"], summary["test_iou"]))

print(f"{'split':<28}{'loss':>10}{'IoU':>10}")
print("-" * 48)
for name, loss, iou in rows:
    print(f"{name:<28}{loss:>10.4f}{iou:>10.4f}")

if "splits" in summary:
    print()
    print("images per split:", summary["splits"])

The test score is the number that matters. It comes from the best-validation checkpoint
evaluated once on images the model never saw during training or epoch selection.

Test IoU landing within a hundredth of validation IoU is the key sanity check: it says
the 70/20/10 split is clean and that choosing the epoch on validation did not quietly
overfit to it.

## Qualitative examples

Numbers hide the failure modes. Below are test-set predictions: the simulated click,
the ground-truth mask, and what the model produced.

In [ ]:
import numpy as np
import torch
import yaml

from src.data.ade20k import discover_samples
from src.data.dataset import ClickSegmentationDataset
from src.data.splits import split_image_paths
from src.model.unet import UNet
from src.training.metrics import iou_score

with open(REPO / "configs/train.yaml") as f:  train_cfg = yaml.safe_load(f)
with open(REPO / "configs/clicks.yaml") as f: click_cfg = yaml.safe_load(f)["clicks"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image_paths = discover_samples(DATA_ROOT)
splits = split_image_paths(image_paths,
                           ratios=tuple(train_cfg["training"]["splits"]),
                           seed=train_cfg["training"]["split_seed"])

test_ds = ClickSegmentationDataset(
    splits["test"],
    image_size=train_cfg["data"]["image_size"],
    click_config=click_cfg,
    deterministic=True,
    lazy=True,
    index_cache=OUTPUTS / "instance_index_test.json",
)

model = UNet(in_channels=5, out_channels=1,
             base_channels=train_cfg["model"]["base_channels"]).to(device)
ckpt = torch.load(OUTPUTS / "checkpoints/best.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"test instances: {len(test_ds)}   checkpoint from epoch {ckpt['epoch']}   device: {device}")

## Is IoU 0.50 any good? Trivial baselines

IoU is not accuracy. In binary classification, random guessing scores 50% and that number
means "no information". Intersection-over-Union does not work that way: a random or
degenerate prediction scores near **zero**, not 0.5.

The cell below measures that directly on the test set, so the model's score can be read
against strategies that contain no information at all.

In [ ]:
from torch.utils.data import DataLoader

# A subsample keeps this quick; IoU is an average so it converges fast.
N_EVAL = 2000
sub = torch.utils.data.Subset(test_ds, list(range(min(N_EVAL, len(test_ds)))))
loader = DataLoader(sub, batch_size=64, shuffle=False, num_workers=4)

def batch_iou(pred_bool, target_bool, eps=1e-6):
    "Mean per-sample IoU for boolean tensors shaped (N,1,H,W)."
    p, t = pred_bool.flatten(1), target_bool.flatten(1)
    inter = (p & t).sum(1).float()
    union = (p | t).sum(1).float()
    return ((inter + eps) / (union + eps)).tolist()

scores = {"model": [], "all foreground": [], "random 50%": [], "disk at click": []}
gen = torch.Generator().manual_seed(0)

with torch.no_grad():
    for inputs, targets in loader:
        t_bool = targets.bool()

        logits = model(inputs.to(device))
        scores["model"] += batch_iou((torch.sigmoid(logits) > 0.5).cpu(), t_bool)

        scores["all foreground"] += batch_iou(torch.ones_like(t_bool), t_bool)

        noise = torch.rand(t_bool.shape, generator=gen) > 0.5
        scores["random 50%"] += batch_iou(noise, t_bool)

        # A filled disk centred on the positive click: the strongest "no learning"
        # baseline, since it at least knows where the user pointed.
        pos = inputs[:, 3:4]
        H, W = pos.shape[-2:]
        yy = torch.arange(H).view(1, 1, H, 1)
        xx = torch.arange(W).view(1, 1, 1, W)
        disks = []
        for b in range(pos.shape[0]):
            ys, xs = torch.nonzero(pos[b, 0], as_tuple=True)
            cy, cx = (ys.float().mean(), xs.float().mean()) if len(ys) else (H / 2, W / 2)
            disks.append(((yy - cy) ** 2 + (xx - cx) ** 2 <= 30 ** 2))
        scores["disk at click"] += batch_iou(torch.cat(disks, 0), t_bool)

print(f"Evaluated on {len(sub)} test instances\n")
print(f"{'strategy':<22}{'mean IoU':>10}")
print("-" * 32)
for name in ["random 50%", "all foreground", "disk at click", "model"]:
    mark = "   <- trained model" if name == "model" else ""
    print(f"{name:<22}{sum(scores[name])/len(scores[name]):>10.4f}{mark}")

In [ ]:
labels = ["random 50%", "all foreground", "disk at click", "model"]
values = [sum(scores[k]) / len(scores[k]) for k in labels]

fig, ax = plt.subplots(figsize=(7, 3.4))
bars = ax.barh(labels, values,
               color=["#bbbbbb", "#bbbbbb", "#bbbbbb", "#c0392b"])
for bar, v in zip(bars, values):
    ax.text(v + 0.008, bar.get_y() + bar.get_height() / 2, f"{v:.3f}",
            va="center", fontsize=10)
ax.set_xlabel("mean IoU on test set")
ax.set_title("Trained model vs strategies that learn nothing")
ax.set_xlim(0, max(values) * 1.25)
ax.grid(axis="x", alpha=.3)
fig.tight_layout()
plt.show()

The comparison is the point. A strategy with no information scores near zero; even one
that is *told where the user clicked* and draws a fixed blob there scores far below the
trained model. The gap between that blob and the model is what the network actually
learned: object extent, boundaries, and which of several overlapping things the click
referred to.

That said, IoU 0.50 is a starting baseline, not a finished result. Published interactive
segmenters reach considerably higher, using pretrained backbones, the full dataset,
higher resolution, and iterative click refinement. This model has none of those yet.

In [ ]:
N_SHOWN = 6
rng = np.random.default_rng(0)
picks = rng.choice(len(test_ds), size=N_SHOWN, replace=False)

fig, axes = plt.subplots(N_SHOWN, 3, figsize=(10.5, 3.3 * N_SHOWN))

for row, idx in enumerate(picks):
    inputs, target = test_ds[int(idx)]
    with torch.no_grad():
        logits = model(inputs.unsqueeze(0).to(device))
    pred = (torch.sigmoid(logits)[0, 0].cpu().numpy() > 0.5)
    iou  = iou_score(logits.cpu(), target.unsqueeze(0))

    image = inputs[:3].permute(1, 2, 0).numpy()
    pos, neg = inputs[3].numpy(), inputs[4].numpy()

    axes[row, 0].imshow(image)
    for chan, colour, marker in ((pos, "lime", "+"), (neg, "red", "x")):
        if chan.any():
            ys, xs = np.nonzero(chan)
            axes[row, 0].scatter(xs.mean(), ys.mean(), c=colour, marker=marker,
                                 s=220, linewidths=3)
    axes[row, 0].set_title("image + simulated click")

    axes[row, 1].imshow(target[0].numpy(), cmap="gray")
    axes[row, 1].set_title("ground truth")

    axes[row, 2].imshow(pred, cmap="gray")
    axes[row, 2].set_title(f"prediction (IoU {iou:.2f})")

    for ax in axes[row]:
        ax.axis("off")

fig.tight_layout()
plt.show()

Green `+` is the positive click identifying the target object; red `x`, where present, is
a negative click just outside it.

## Where this stands, and what would improve it

The model reliably finds the clicked object and gets its rough extent right. Errors
concentrate in two places: boundaries are soft rather than crisp, and large objects with
ambiguous edges (road against sidewalk, building against sky) bleed into their
neighbours.

Ranked by expected benefit:

1. **More data.** Training used 3,000 of the 25,574 available images, about 12%. This is
   the clearest limitation, and the curves support it: the model is data-starved rather
   than capacity-starved or under-trained.
2. **A pretrained encoder.** Training from scratch was a deliberate project requirement,
   but an ImageNet-pretrained backbone is what most published interactive segmenters use
   and would likely help most after data.
3. **Higher resolution.** At 128x128 fine boundaries are barely representable. Only 0.24%
   of instances vanish at this size, so this is about boundary precision, not coverage.
4. **Iterative clicks.** Real users click repeatedly to correct a mask. Training with a
   single click optimises for a task easier than the one the tool actually performs.

Not yet measured: **NoC** (number of clicks to reach a target IoU), the standard
interactive-segmentation metric, and a comparison against a pretrained SAM baseline.